# Download Data From Kaggle

For more information about the competition check the [link](https://www.kaggle.com/competitions/titanic/)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    # Setup Kaggle API
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

    !mkdir -p ~/.kaggle
    kaggle_config = {"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download data if it doesn't exist yet
    if not os.path.exists('data'):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists('data'):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print('Data already exists')

# Exploring The Dataset

In [ ]:
import pandas as pd

df = pd.read_csv('data/train.csv')

df.head()

### Missing values

In [ ]:
print("Missing values in the dataset:\n"+"="*20)
print(df.isna().sum())
print("="*20)
df[df.isna().any(axis=1)]

### Filling mising values

In [ ]:
mean_age = df['Age'].mean()
df['Age'] = df['Age'].fillna(mean_age)
df['Embarked'] = df['Embarked'].fillna('S')
df['Cabin'] = df['Cabin'].ffill()
df.isna().sum()

## Statistics

In [ ]:
total_passengers = df.shape[0]
survived_passengers = df[df["Survived"] == 1].shape[0]

print(f"total number of passengers from southampton: {df[df['Embarked'] == 'S'].shape[0]}")
print(f"total number of passengers from cherbourg: {df[df['Embarked'] == 'C'].shape[0]}")
print(f"total number of passengers from queenstown: {df[df['Embarked'] == 'Q'].shape[0]}")

## Remove unsed columns

In [ ]:
df.drop(['Embarked', 'Cabin', 'Ticket', 'Name', 'PassengerId', 'Fare'], axis=1, inplace=True)
df.head()

In [ ]:
Y = df['Survived']
X = df.drop('Survived', axis=1)

In [ ]:
Y = df['Survived']
X = df.drop(["Survived"], axis=1)

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import RandomizedSearchCV, KFold

from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    [
        ("lr", LogisticRegression()),
        ("svm", SVC()),
    ]
)

param_distribution = [
    {
        "svm__C": [0.1, 0.5, 1, 1.5, 2, 5, 10], 
        "svm__kernel": ["poly", "linear", "rbf"],
        "svm__degree": [2, 3, 4, 5, 6, 7] 
    },
    {
        "lr__C": [0.1, 0.5, 1, 1.5, 2, 5, 10],
        "lr__solver": ["saga", "lbfgs"]
    }
]

search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distribution, verbose=1)

search.fit(X, Y)
print(search.best_score_)